In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

MOVEMENT_DIR = Path(
    r"C:\Users\Sheel\OneDrive\Documents\Default Project\data\pads-parkinsons-disease-smartwatch-dataset-1.0.0\pads-parkinsons-disease-smartwatch-dataset-1.0.0\preprocessed\movement"
)

file_list = pd.read_csv(MOVEMENT_DIR.parent / "file_list.csv")

tasks = [
    "Relaxed1", "Relaxed2", "RelaxedTask1", "RelaxedTask2",
    "StretchHold", "HoldWeight", "DrinkGlas", "CrossArms",
    "TouchNose", "Entrainment1", "Entrainment2",
]
wrists = ["LeftWrist", "RightWrist"]
sensors = ["Accelerometer", "Gyroscope"]
axes = ["X", "Y", "Z"]
channels = [f"{t}_{w}_{s}_{a}" for t in tasks for w in wrists for s in sensors for a in axes]

print(len(channels), "channels")
print(file_list["condition"].value_counts().to_string())

132 channels
condition
Parkinson's                 276
Healthy                      79
Other Movement Disorders     60
Essential Tremor             28
Atypical Parkinsonism        15
Multiple Sclerosis           11


In [2]:
bin_files = sorted(MOVEMENT_DIR.glob("*_ml.bin"))
ids = np.array([int(f.name.split("_")[0]) for f in bin_files])
movement = np.stack([np.fromfile(f, dtype=np.float32).reshape(len(channels), -1) for f in bin_files])
print("movement:", movement.shape, movement.dtype)
print("ids aligned with file_list:", np.array_equal(ids, file_list["id"].values))

movement: (469, 132, 976) float32
ids aligned with file_list: True


In [3]:
channel_df = pd.DataFrame({
    "channel": channels,
    "task": [c.split("_")[0] for c in channels],
    "wrist": [c.split("_")[1] for c in channels],
    "sensor": [c.split("_")[2] for c in channels],
    "axis": [c.split("_")[3] for c in channels],
})

stream_feats = []
feature_names = []
for (task, wrist), grp in channel_df.groupby(["task", "wrist"]):
    acc = movement[:, grp.index[grp["sensor"] == "Accelerometer"].tolist(), :]
    gyro = movement[:, grp.index[grp["sensor"] == "Gyroscope"].tolist(), :]
    acc_norm = np.linalg.norm(acc, axis=1)
    gyro_norm = np.linalg.norm(gyro, axis=1)
    jerk = np.diff(acc_norm, axis=1)
    for name, sig in [("acc_norm", acc_norm), ("gyro_norm", gyro_norm), ("jerk", jerk)]:
        stream_feats.append(sig.mean(axis=1))
        stream_feats.append(sig.std(axis=1))
        feature_names += [f"{task}_{wrist}_{name}_mean", f"{task}_{wrist}_{name}_std"]

X = np.column_stack(stream_feats)
conditions = ["Healthy", "Parkinson's", "Essential Tremor", "Atypical Parkinsonism", "Multiple Sclerosis"]
keep = np.isin(file_list["condition"].to_numpy(), conditions)
X = X[keep]
y = pd.Categorical(file_list.loc[keep, "condition"].to_numpy(), categories=conditions).codes
print("X:", X.shape, "| y:", y.shape)

X: (409, 132) | y: (409,)


In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

binary = np.isin(y, [0, 1])
Xb, yb = X[binary], y[binary]
yb = (yb == 1).astype(int)

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    Xb, yb, test_size=0.3, stratify=yb, random_state=0
)

model = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(solver="saga", l1_ratio=0.5, max_iter=5000, random_state=0)),
])
model.fit(Xb_train, yb_train)

print("PD vs HC accuracy:", accuracy_score(yb_test, model.predict(Xb_test)))
print(classification_report(yb_test, model.predict(Xb_test), target_names=["HC", "Parkinson's"]))

PD vs HC accuracy: 0.7289719626168224
              precision    recall  f1-score   support

          HC       0.35      0.25      0.29        24
 Parkinson's       0.80      0.87      0.83        83

    accuracy                           0.73       107
   macro avg       0.58      0.56      0.56       107
weighted avg       0.70      0.73      0.71       107



In [5]:
target_names = ["HC", "Parkinson's", "Essential Tremor", "Atypical PD", "MS"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=0
)

multi = Pipeline([
    ("scale", StandardScaler()),
    ("clf", LogisticRegression(solver="saga", l1_ratio=0.5, max_iter=5000, random_state=0)),
])
multi.fit(X_train, y_train)

print("5-class accuracy:", accuracy_score(y_test, multi.predict(X_test)))
print(classification_report(y_test, multi.predict(X_test), target_names=target_names))

5-class accuracy: 0.6585365853658537
                  precision    recall  f1-score   support

              HC       0.36      0.33      0.35        24
     Parkinson's       0.75      0.83      0.79        83
Essential Tremor       0.50      0.38      0.43         8
     Atypical PD       0.50      0.20      0.29         5
              MS       0.00      0.00      0.00         3

        accuracy                           0.66       123
       macro avg       0.42      0.35      0.37       123
    weighted avg       0.63      0.66      0.64       123

